### Step 1: Install necesscary packages

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

Defaulting to user installation because normal site-packages is not writeable
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 9.5 MB/s eta 0:00:01
   ---------------- ----------------------- 3.9/9.3 MB 9.8 MB/s eta 0:00:01
   ------------------------- -------------- 6.0/9.3 MB 10.2 MB/s eta 0:00:01
   ----------------------------------- ---- 8.4/9.3 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 10.3 MB/s eta 0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   -------------------- ------------------- 1.3/2.5 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 7.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.6 MB 9.7 MB


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Package imports and configuration

In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
if torch.cuda.is_available():
    available = 'cuda'
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    available = 'mps'
else:
    available = 'cpu'
device = available
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [2]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [3]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Generating Dataset

In [4]:

import random
import json

random.seed(42)

data = []
seen_prompts = set()

# Define the 12 equation types
equation_types = [
    'addition', 'subtraction', 'multiplication', 'division',
    'addition_unknown', 'addition2_unknown',
    'subtraction_unknown', 'subtraction2_unknown',
    'multiplication_unknown', 'multiplication2_unknown',
    'division_unknown', 'division2_unknown'
]

# Allocate samples across arithmetic and algebra equations
samples_per_type = {
    'addition': 17000,
    'subtraction': 12000,
    'multiplication': 3500,
    'division': 4000,
    'addition_unknown': 14000,
    'addition2_unknown': 14000,
    'subtraction_unknown': 14000,
    'subtraction2_unknown': 12000,
    'multiplication_unknown': 2500,
    'multiplication2_unknown': 2500,
    'division_unknown': 2500,
    'division2_unknown': 2500
}

print("Total target samples:", sum(samples_per_type.values()))
print("Generating positive-negative pairs...\n")

# Generate samples for each equation type
for eq_type in equation_types:
    generated = 0
    attempts = 0
    target_samples = samples_per_type[eq_type]
    max_attempts = target_samples * 20

    while generated < target_samples and attempts < max_attempts:
        attempts += 1

        if eq_type == 'addition':
            a = random.randint(0, 199)
            b = random.randint(0, 199)
            prompt = f"{a}+{b}=?"
            answer = a + b
            positive = f"The answer is {answer} because {a}+{b} equals {answer}."

        elif eq_type == 'subtraction':
            a = random.randint(0, 199)
            b = random.randint(0, 199)

            if b > a:
                a, b = b, a

            prompt = f"{a}-{b}=?"
            answer = a - b
            positive = f"The answer is {answer} because {a}-{b} equals {answer}."

        elif eq_type == 'multiplication':
            a = random.randint(1, 20)
            b = random.randint(0, 200)
            prompt = f"{a}*{b}=?"
            answer = a * b
            positive = f"The answer is {answer} because {a}*{b} equals {answer}."

        elif eq_type == 'division':
            b = random.randint(1, 30)
            answer = random.randint(0, 200)
            a = b * answer
            prompt = f"{a}/{b}=?"
            positive = f"The answer is {answer} because {a}/{b} equals {answer}."

        elif eq_type == 'addition_unknown':
            answer = random.randint(0, 199)
            a = random.randint(0, 199)
            b = a + answer
            prompt = f"{a}+x={b},x=?"
            positive = f"The answer is {answer} because {b}-{a} equals {answer}."

        elif eq_type == 'addition2_unknown':
            a = random.randint(0, 199)
            answer = random.randint(0, 199)
            b = answer + a
            prompt = f"x+{a}={b},x=?"
            positive = f"The answer is {answer} because {b}-{a} equals {answer}."

        elif eq_type == 'subtraction_unknown':
            answer = random.randint(0, 199)
            b = random.randint(0, 199)
            a = answer + b
            prompt = f"{a}-x={b},x=?"
            positive = f"The answer is {answer} because {a}-{b} equals {answer}."

        elif eq_type == 'subtraction2_unknown':
            answer = random.randint(0, 199)
            a = random.randint(0, answer)
            b = answer - a
            prompt = f"x-{a}={b},x=?"
            positive = f"The answer is {answer} because {b}+{a} equals {answer}."

        elif eq_type == 'multiplication_unknown':
            a = random.randint(1, 20)
            answer = random.randint(0, 200)
            b = a * answer
            prompt = f"{a}*x={b},x=?"
            positive = f"The answer is {answer} because {b}/{a} equals {answer}."

        elif eq_type == 'multiplication2_unknown':
            a = random.randint(1, 20)
            answer = random.randint(0, 200)
            b = a * answer
            prompt = f"x*{a}={b},x=?"
            positive = f"The answer is {answer} because {b}/{a} equals {answer}."

        elif eq_type == 'division_unknown':
            answer = random.randint(1, 20)
            b = random.randint(1, 200)
            a = answer * b
            prompt = f"{a}/x={b},x=?"
            positive = f"The answer is {answer} because {a}/{b} equals {answer}."

        elif eq_type == 'division2_unknown':
            a = random.randint(1, 20)
            b = random.randint(0, 200)
            answer = b * a
            prompt = f"x/{a}={b},x=?"
            positive = f"The answer is {answer} because {a}*{b} equals {answer}."

        # Skip duplicate questions
        if prompt in seen_prompts:
            continue

        seen_prompts.add(prompt)

        negative = "Sorry, I do not know."

        data.append({
            "negative": prompt + " " + negative,
            "positive": prompt + " " + positive
        })

        generated += 1

    # Warn if target is not reached
    if generated < target_samples:
        print(
            f"WARNING: {eq_type} came up short: "
            f"{generated}/{target_samples}"
        )

    print(f"✓ {eq_type}: {generated} samples generated")

print(f"\n✓ Total generated: {len(data)} unique pairs")

# Shuffle samples to mix equation types
random.shuffle(data)

# Save to JSON
with open("pos_neg_pairs.json", "w") as f:
    json.dump(data, f)

print("✓ Saved to pos_neg_pairs.json")

# Validate dataset size and uniqueness
assert len(data) == sum(samples_per_type.values()), \
    "Dataset size does not match the target."

assert len(seen_prompts) == len(data), \
    "Duplicate prompts detected."

print("✓ Dataset size and uniqueness validated")


Total target samples: 100500
Generating positive-negative pairs...

✓ addition: 17000 samples generated
✓ subtraction: 12000 samples generated
✓ multiplication: 3500 samples generated
✓ division: 4000 samples generated
✓ addition_unknown: 14000 samples generated
✓ addition2_unknown: 14000 samples generated
✓ subtraction_unknown: 14000 samples generated
✓ subtraction2_unknown: 12000 samples generated
✓ multiplication_unknown: 2500 samples generated
✓ multiplication2_unknown: 2500 samples generated
✓ division_unknown: 2500 samples generated
✓ division2_unknown: 2500 samples generated

✓ Total generated: 100500 unique pairs
✓ Saved to pos_neg_pairs.json
✓ Dataset size and uniqueness validated


### Step 5: Load Data (**students are required to complete this part!**)

A possible issue: stoi.keys does not include '!', thus '!' cannot be used in the data pairs of pos_neg_pairs.json.

In [5]:
print(stoi.keys())

dict_keys(['\n', ' ', "'", '*', '+', ',', '-', '.', '/', '=', '?', '’', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'])


Here, when reading and loading lines from the json file, the data is cleaned in case it contains an invalid key like '!'. 

In [6]:

with open("pos_neg_pairs.json") as f:
    lines = json.load(f)
    #### check our data here
    print(type(lines))
    print("Total number of pairs:", len(lines))
    for item in random.sample(lines,5): 
        print(item['negative'])
        print(item['positive'])
        print("-----")

#### clean data just in case
for pair in lines:
    for key in ['negative', 'positive']:
        pair[key] = "".join(char for char in pair[key] if char in stoi)

<class 'list'>
Total number of pairs: 100500
259-x=76,x=? Sorry, I do not know.
259-x=76,x=? The answer is 183 because 259-76 equals 183.
-----
199-98=? Sorry, I do not know.
199-98=? The answer is 101 because 199-98 equals 101.
-----
177-68=? Sorry, I do not know.
177-68=? The answer is 109 because 177-68 equals 109.
-----
x*9=702,x=? Sorry, I do not know.
x*9=702,x=? The answer is 78 because 702/9 equals 78.
-----
199+60=? Sorry, I do not know.
199+60=? The answer is 259 because 199+60 equals 259.
-----


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

For the optimiser, AdamW uses adaptive learning rates for individual parameters and applies decoupled weight decay. During fine-tuning, overfitting can be reduced through this regularisation. For the scheduler, StepLR gradually reduces learning rate during training. The learning rate is halved every two epochs. Larger parameters are allowed at the start, but updates become smaller to support stable convergence.

In [7]:
# recommend to use the AdamW optimizer 
optimiser = torch.optim.AdamW(gpt.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimiser, step_size=2, gamma=0.5)

### Step 7: Begin training (**students are required to complete this part!**)

This simply checks for MPS device in the case that a Macbook is used.

In [8]:
print("Selected device:", device)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

#### check for mps on macbook

Selected device: cpu
CUDA available: False
MPS available: False
MPS built: False



During training, the model will be fine-tuned using batches of positive and negative responses. Positive responses are preferred, and negative responses are rejected. The log probabilities of each batch are calculated and used to compute a preference-based loss to encourage the model to favour more positive (correct) responses. The loss is backpropagated and AdamW updates model parameters. StepLR reduces learning rate every 2 epochs, and the average training loss is recorded so that training progress can be visualised.

In [ ]:
total_steps = len(lines) // batch_size
#### 
# to monitor loss
epoch_loss = []
####
for epoch in range(epochs):
    ####
    # to monitor loss
    loss_history = []
    ####
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        
        optimiser.zero_grad()

        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        loss.backward()

        optimiser.step()

        loss_val = loss.item()
        loss_history.append(loss_val)

    avg_loss = sum(loss_history) / len(loss_history)
    epoch_loss.append(avg_loss)
    scheduler.step()
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

#### here we print the loss...
epoch_no = [1,2,3,4,5]
plt.figure()
plt.plot(epoch_no, epoch_loss, label="Loss against epoch")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.xlim(left=0)
plt.ylim(bottom=0)
plt.show()

### Step 8: Begin testing (**students are required to complete this part!**)

### 8.1 Evaluation Dataset Preparation

Generate 100 unique test questions across five mathematical categories:
addition, subtraction, multiplication, division, and algebra.

Each category contains 20 questions. Questions appearing in the
training dataset are excluded to evaluate performance on unseen examples.

In [ ]:
import random

random.seed(42)

candidate_questions = []

for _ in range(200):
    a = random.randint(1, 200)
    b = random.randint(1, 200)
    candidate_questions.append(f"{a}+{b}=?")

print("Generated:", len(candidate_questions))
print(candidate_questions[:5])

training_questions = {
    item["positive"].split("?")[0] + "?"
    for item in lines
}

unseen_candidates = [
    q for q in candidate_questions
    if q not in training_questions
]

print("Unseen candidates:", len(unseen_candidates))

addition_test_set = list(dict.fromkeys(unseen_candidates))[:20]

print("Unique unseen addition questions:", len(addition_test_set))

subtraction_candidates = []

for _ in range(200):
    a = random.randint(1, 200)
    b = random.randint(1, a)
    subtraction_candidates.append(f"{a}-{b}=?")

subtraction_test_set = list(dict.fromkeys(
    q for q in subtraction_candidates
    if q not in training_questions
))[:20]

print("Unique unseen subtraction questions:", len(subtraction_test_set))

multiplication_candidates = []

for _ in range(200):
    a = random.randint(1, 20)
    b = random.randint(1, 200)
    multiplication_candidates.append(f"{a}*{b}=?")

multiplication_test_set = list(dict.fromkeys(
    q for q in multiplication_candidates
    if q not in training_questions
))[:20]

print("Unique unseen multiplication questions:", len(multiplication_test_set))

division_candidates = []

for _ in range(200):
    divisor = random.randint(1, 20)
    answer = random.randint(1, 200)
    dividend = divisor * answer
    division_candidates.append(f"{dividend}/{divisor}=?")

division_test_set = list(dict.fromkeys(
    q for q in division_candidates
    if q not in training_questions
))[:20]

print("Unique unseen division questions:", len(division_test_set))

algebra_candidates = []

for _ in range(200):
    x = random.randint(1, 200)
    a = random.randint(1, 200)
    algebra_candidates.append(f"x+{a}={x+a},x=?")

algebra_test_set = list(dict.fromkeys(
    q for q in algebra_candidates
    if q not in training_questions
))[:20]

print("Unique unseen algebra questions:", len(algebra_test_set))

evaluation_test_set = (
    addition_test_set
    + subtraction_test_set
    + multiplication_test_set
    + division_test_set
    + algebra_test_set
)

print("Total evaluation questions:", len(evaluation_test_set))

Generated: 200
['164+29=?', '7+190=?', '71+63=?', '58+36=?', '189+27=?']
Unseen candidates: 102
Unique unseen addition questions: 20
Unique unseen subtraction questions: 20
Unique unseen multiplication questions: 20
Unique unseen division questions: 20
Unique unseen algebra questions: 20
Total evaluation questions: 100


### 8.2 Model Testing

Load the fine-tuned NanoGPT model and evaluate its performance
on the 100 unseen mathematical questions.

A fixed random seed is used for reproducibility.
The model's generated responses are stored for subsequent evaluation.

In [31]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
test_results = []
with torch.no_grad():
    for prompt in evaluation_test_set: 
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y = gpt.generate(x, max_new_tokens=100)
        response = decode(y[0][0].tolist())
        print(response)
        test_results.append({
            "question": prompt,
            "response": response
        })

71+63=? The answer is 134 because 71+63 equals 134.
189+27=? The answer is 216 because 189+27 equals 216.
174+190=? The answer is 364 because 174+190 equals 364.
140+23=? The answer is 163 because 140+23 equals 163.
152+109=? The answer is 261 because 152+109 equals 261.
9+8=? The answer is 17 because 9+8 equals 17.
144+51=? The answer is 195 because 144+51 equals 195.
108+57=? The answer is 165 because 108+57 equals 165.
98+25=? The answer is 123 because 98+25 equals 123.
155+68=? The answer is 223 because 155+68 equals 223.
12+187=? The answer is 209 because 12+187 equals 209.
32+97=? The answer is 139 because 32+97 equals 139.
76+161=? The answer is 237 because 76+161 equals 237.
159+93=? The answer is 252 because 159+93 equals 252.
181+18=? The answer is 199 because 181+18 equals 199.
98+72=? The answer is 160 because 98+72 equals 160.
117+163=? The answer is 290 because 117+163 equals 290.
95+91=? The answer is 186 because 95+91 equals 186.
69+180=? The answer is 249 because 69+18

In [32]:
print("Total responses:", len(test_results))
print("First result:", test_results[0])

Total responses: 100
First result: {'question': '71+63=?', 'response': '71+63=? The answer is 134 because 71+63 equals 134.'}


### 8.3 Answer Extraction and Validation

Extract the numerical answer from each response generated by NanoGPT.

Calculate the correct answer for each question and store both values
for comparison during accuracy evaluation.

In [33]:
import re

for result in test_results:
    match = re.search(r"The answer is (-?\d+)", result["response"])
    result["predicted_answer"] = int(match.group(1)) if match else None

print("First extracted answer:", test_results[0]["predicted_answer"])

First extracted answer: 134


In [34]:
def get_correct_answer(question):
    if "x" in question:
        equation = question.split(",")[0]
        left, right = equation.split("=")
        a = int(left.split("+")[1])
        b = int(right)
        return b - a

    expression = question.replace("=?", "")
    return eval(expression)

for result in test_results:
    result["correct_answer"] = get_correct_answer(result["question"])

print(test_results[0])

{'question': '71+63=?', 'response': '71+63=? The answer is 134 because 71+63 equals 134.', 'predicted_answer': 134, 'correct_answer': 134}


### 8.4 Accuracy Evaluation

Compare NanoGPT's predicted answers against the correct answers.

Calculate the overall accuracy across all 100 test questions,
followed by individual accuracy scores for addition, subtraction,
multiplication, division, and algebra.

In [35]:
correct_count = sum(
    result["predicted_answer"] == result["correct_answer"]
    for result in test_results
)

total = len(test_results)
accuracy = correct_count / total * 100

print(f"Correct answers: {correct_count}/{total}")
print(f"Accuracy: {accuracy:.2f}%")

Correct answers: 56/100
Accuracy: 56.00%


In [36]:
categories = ["Addition", "Subtraction", "Multiplication", "Division", "Algebra"]

for i, category in enumerate(categories):
    results = test_results[i*20:(i+1)*20]

    correct = sum(
        r["predicted_answer"] == r["correct_answer"]
        for r in results
    )

    print(f"{category}: {correct}/20 ({correct/20*100:.1f}%)")

Addition: 16/20 (80.0%)
Subtraction: 14/20 (70.0%)
Multiplication: 5/20 (25.0%)
Division: 4/20 (20.0%)
Algebra: 17/20 (85.0%)
